# Coupled Power–Heat Network Controller Tutorial

This tutorial demonstrates how multinet controllers coordinate coupled electrical and heating networks using **pandapower** and **pandapipes**. Run the cells from top to bottom.

## 1. Import the required packages

We use the example power network from pandapower, build a water-based heating network with pandapipes, and then connect them in a multinet. The coupling controllers exchange power and heat between the networks.

In [1]:
import pandapipes as pp
import pandapower as ppower
from pandapower import networks as pandasnet
from pandapipes.multinet.create_multinet import create_empty_multinet, add_net_to_multinet
from pandapipes.multinet.control.run_control_multinet import run_control
from pandapipes.multinet.control.multinet_control_power2heat import (
    P2HControlMultiEnergy, H2PControlMultiEnergy
)

## 2. Create the electrical network

Start with `pandapower.networks.example_simple()`, a small pre-defined electrical network. The later conversion elements will be connected to bus 6 in this example.

In [2]:
power_net = pandasnet.example_simple()
power_net.bus.head()

## 3. Build the heating network

Create an empty pandapipes network with water as the fluid, then add four junctions. The junction parameters specify nominal pressure and starting temperature.

In [3]:
heat_net = pp.create_empty_network(fluid="water")

j0 = pp.create_junction(heat_net, pn_bar=5, tfluid_k=293.15, name="junction 0")
j1 = pp.create_junction(heat_net, pn_bar=5, tfluid_k=293.15, name="junction 1")
j2 = pp.create_junction(heat_net, pn_bar=5, tfluid_k=293.15, name="junction 2")
j3 = pp.create_junction(heat_net, pn_bar=5, tfluid_k=293.15, name="junction 3")
heat_net.junction

### 3.1 Add pump, heat exchanger, and pipes

The circulation pump drives the water flow, the heat exchanger extracts heat, and the two pipes connect the loop. These parameters match the original example.

In [4]:
pp.create_circ_pump_const_mass_flow(
    heat_net, return_junction=j3, flow_junction=j0,
    p_flow_bar=5, mdot_flow_kg_per_s=20, t_flow_k=273.15 + 35
)
pp.create_heat_exchanger(
    heat_net, from_junction=j1, to_junction=j2,
    diameter_m=200e-3, qext_w=100000
)
pp.create_pipe_from_parameters(
    heat_net, from_junction=j0, to_junction=j1, length_km=1,
    diameter_m=200e-3, k_mm=.1, alpha_w_per_m2k=10,
    sections=5, text_k=283
)
pp.create_pipe_from_parameters(
    heat_net, from_junction=j2, to_junction=j3, length_km=1,
    diameter_m=200e-3, k_mm=.1, alpha_w_per_m2k=10,
    sections=5, text_k=283
)

## 4. Combine the two networks

A **multinet** holds both networks. The names `power` and `heat` are used later when creating the coupling controllers.

In [5]:
multinet = create_empty_multinet("multinet")
add_net_to_multinet(multinet, power_net, "power")
add_net_to_multinet(multinet, heat_net, "heat")

## 5. Add power-to-heat and heat-to-power elements

For **power-to-heat (P2H)**, an electrical load is paired with a heating-network exchanger. For **heat-to-power (H2P)**, a heating-network exchanger is paired with an electrical static generator. The controllers added in the next step will link the corresponding values.

In [6]:
# Power-to-heat elements
p2h_id_el = ppower.create_load(
    power_net, bus=6, p_mw=.0002, name="power to heat consumption"
)
p2h_id_heat = pp.create_heat_exchanger(
    heat_net, from_junction=0, to_junction=1,
    diameter_m=200e-3, qext_w=0, name="power to heat feed in"
)

# Heat-to-power elements
h2p_id_heat = pp.create_heat_exchanger(
    heat_net, from_junction=2, to_junction=3,
    diameter_m=200e-3, qext_w=200000, name="power to heat feed in"
)
h2p_id_el = ppower.create_sgen(
    power_net, bus=6, p_mw=0, name="fuel cell feed in"
)

## 6. Configure the coupling controllers

The P2H controller links the electrical load to the thermal exchanger. The H2P controller links the thermal exchanger to the generator. Their `efficiency` values are the same as in the original example.

In [7]:
p2h_ctrl = P2HControlMultiEnergy(
    multinet, p2h_id_el, p2h_id_heat, efficiency=3,
    name_power_net="power", name_heat_net="heat"
)
h2p_ctrl = H2PControlMultiEnergy(
    multinet, h2p_id_el, h2p_id_heat, efficiency=2,
    name_power_net="power", name_heat_net="heat"
)

## 7. Check the starting values

Before executing the controllers, inspect the heat exchanged by the P2H element (watts) and the generated electrical power from the H2P element (MW).

In [8]:
print("P2H heat before control (W):", heat_net.heat_exchanger.loc[p2h_id_heat, "qext_w"])
print("H2P power before control (MW):", power_net.sgen.loc[h2p_id_el, "p_mw"])

## 8. Run the multinet control simulation

`run_control` applies the configured controllers and calculates the coupled network state.

In [9]:
run_control(multinet)

## 9. Inspect the results

Compare these values with the starting values above. The updated values illustrate how the controllers couple thermal and electrical elements.

In [10]:
print("P2H heat after control (W):", heat_net.heat_exchanger.loc[p2h_id_heat, "qext_w"])
print("H2P power after control (MW):", power_net.sgen.loc[h2p_id_el, "p_mw"])